In [ ]:
import pandas as pd
import numpy as np
import anndata
import json
import os
import scanpy as sc
from sklearn.model_selection import train_test_split

## Data inspection

In [ ]:
# 1. Read the data
adata = anndata.read_h5ad("../data_centralized/pancreas_train.h5ad")

In [ ]:
# 2. Display the AnnData object summary
print("AnnData object summary:")
print(adata)

In [ ]:
# 3. Display the first few rows of the observation metadata
print("\nFirst 5 rows of adata.obs:")
print(adata.obs.head())

In [ ]:
# 4. Display available layers
print("\nAvailable layers in adata:")
print(adata.layers.keys())

In [ ]:
# 5. Display the first 5x5 block of the counts layer (if it exists)
print("\nFirst 5x5 of counts layer:")
print(pd.DataFrame(adata.layers["counts"][:5, :5], 
                    columns=adata.var_names[:5], 
                    index=adata.obs_names[:5]))

In [ ]:
# 6. How many unique technologies are present, and what are their names?

techs = adata.obs['tech'].unique()
print(f"Number of unique technologies: {len(techs)}")
print("Technologies:", ", ".join(techs))

In [ ]:
# 7. What is the total number of genes measured in the dataset?

num_genes = adata.shape[1]
print(f"\nTotal number of genes measured: {num_genes}")

In [ ]:
# 8. What is the total number of samples (cells) in the dataset?

num_samples = adata.shape[0]
print(f"\nTotal number of samples (cells): {num_samples}")

In [ ]:
# 9. How many samples (cells) belong to each technology?

tech_counts = adata.obs['tech'].value_counts()
print("\nNumber of samples (cells) per technology:")
print(tech_counts.to_string())

## Variance analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

# Compute per-gene mean and variance from raw counts
mean_expr = np.array(adata.X.mean(axis=0)).flatten()
# For sparse matrices:
var_expr   = np.array(adata.X.var(axis=0)).flatten()

# Run HVG to get the normalized dispersion scores
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat', inplace=True)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: raw variance vs mean ---
ax = axes[0]
hvg_mask = adata.var['highly_variable']
ax.scatter(mean_expr[~hvg_mask], var_expr[~hvg_mask], s=2, alpha=0.3, color='grey', label='other')
ax.scatter(mean_expr[hvg_mask],  var_expr[hvg_mask],  s=2, alpha=0.6, color='tomato', label='HVG')
ax.set_xlabel('Mean expression')
ax.set_ylabel('Variance')
ax.set_title('Raw variance vs mean')
ax.legend(markerscale=4)

# --- Right: same but log-log scale to see the trend clearly ---
ax = axes[1]
ax.scatter(mean_expr[~hvg_mask], var_expr[~hvg_mask], s=2, alpha=0.3, color='grey', label='other')
ax.scatter(mean_expr[hvg_mask],  var_expr[hvg_mask],  s=2, alpha=0.6, color='tomato', label='HVG')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Mean expression (log)')
ax.set_ylabel('Variance (log)')
ax.set_title('Raw variance vs mean (log-log)')
ax.legend(markerscale=4)

plt.tight_layout()
plt.show()

## HVG identification

### Built-in function

In [ ]:
from app.task import compute_hvg_client_stats, compute_hvg_from_stats

import json
from typing import List, Dict

# Load the all genes list
with open("../data_centralized/all_genes.json", "r") as f:
    all_genes_list = json.load(f)


In [ ]:
# Built-in on full adata (without batch_key to match custom, which doesn't account for batches)
builtin_df = sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False, inplace=False)
builtin_df.head()

### With custom functions (two-step process)

In [ ]:
stats = compute_hvg_client_stats(adata, all_genes_list)
print(stats.keys()) 
print("Sum:", stats['sum'][:5])
print("Sum_sq:", stats['sum_sq'][:5])
print("Gene names:", stats['gene_names'][:5]) 
print("n_cells:", stats['n_cells'])

ours_df =compute_hvg_from_stats(stats['sum'], stats['sum_sq'], stats['n_cells'], stats['gene_names'])
ours_df.head()

### Additivity of the process based on custom functions

In [ ]:
# Splitting into two adatas
n_cells = adata.n_obs
idx = np.random.permutation(n_cells)
split = n_cells // 2

adata_1 = adata[idx[:split]].copy()
adata_2 = adata[idx[split:]].copy()

In [ ]:
# Stats for both 
stats_1 = compute_hvg_client_stats(adata_1, all_genes_list)
stats_2 = compute_hvg_client_stats(adata_2, all_genes_list)

# Retrieving global HVGs
ours_additive_df =compute_hvg_from_stats(
    stats_1['sum']+stats_2['sum'], 
    stats_1['sum_sq']+stats_2['sum_sq'], 
    stats_1['n_cells']+stats_2['n_cells'], 
    stats_1['gene_names'] # any of the two will work here
    )
ours_additive_df.head()

### Final save

In [ ]:
# Save the top 2000 genes from built-in
top2000_genes = builtin_df[builtin_df['highly_variable']].index.tolist()
with open("../data_centralized/top2k_genes.json", "w") as f:
    json.dump(top2000_genes, f, indent=2)
print(f"Saved {len(top2000_genes)} top genes to ../data_centralized/top2k_genes.json")
